> ## SOLUTION / ANSWER KEY &mdash; Lab 5.5
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-05-tool-routing.ipynb`](../lab-05-tool-routing.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.5 &mdash; Tool Routing: Dispatch an Action to a Tool

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Look a tool up by name in the registry and run it
- Handle an unknown tool AND a failing tool without crashing
- Route a REAL action the model chose through your safe dispatcher

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Anatomy of an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-05"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Once the model has chosen an **action name**, the orchestrator must **route** it to the matching tool, run
it, and wrap the result as an **observation**. Real models hallucinate tool names and feed bad inputs, so
routing must fail **safely**: an unknown or failing tool returns a message, not a crash. The `try/except`
is what turns a crash into a recoverable observation &mdash; genuine plumbing, not an LLM stand-in.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
KNOWLEDGE = {"capital of france": "Paris", "population of metropolis": "120000"}
REGISTRY = {
    "calculator": {"name": "calculator", "fn": safe_calc},
    "lookup":     {"name": "lookup", "fn": lambda k: KNOWLEDGE.get(k.lower().strip(), "unknown")},
    "weather":    {"name": "weather", "fn": lambda city: "sunny 24C in " + str(city).strip()},
}
print("registry:", list(REGISTRY))
# Note: safe_calc RAISES on non-math input -- your router must survive that, not crash.

## Build it
Implement `route`: find the tool, run it, return a dict &mdash; surviving an **unknown** tool name and a
tool that **raises**. Test it on all three hazards first.

In [ ]:
def route(registry, action, arg):
    tool = registry.get(action)
    if tool is None:
        return {"ok": False, "observation": "unknown tool: " + str(action)}
    try:
        result = tool["fn"](arg)
        return {"ok": True, "observation": result}
    except Exception as e:
        return {"ok": False, "observation": "tool error: " + type(e).__name__}

try:
    print(route(REGISTRY, "calculator", "10/2"))
    print(route(REGISTRY, "weather", "tokyo"))
    print(route(REGISTRY, "no_such_tool", "x"))       # a hallucinated name -> handled, no crash
    print(route(REGISTRY, "calculator", "not math"))  # a tool that RAISES -> caught, no crash
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Let the **real** model pick a tool + input for a task, then route *its* real choice through your safe dispatcher.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        question = "what's the weather in Tokyo right now?"
        ask = ("Tools you may use: calculator, lookup, weather.\n"
               "For the task below reply EXACTLY two lines and nothing else:\n"
               "TOOL: <one tool name>\nINPUT: <the tool input>\n\n"
               "Task: " + question)
        reply = llm_text(ask)
        print("MODEL SAID:\n" + reply)
        def _line(txt, tag):
            for ln in txt.splitlines():
                if ln.strip().lower().startswith(tag.lower() + ":"):
                    return ln.split(":", 1)[1].strip()
            return ""
        action, arg = _line(reply, "TOOL").lower(), _line(reply, "INPUT")
        print("---")
        print("model chose action:", repr(action), "| input:", repr(arg))
        print("routed through YOUR safe router ->", route(REGISTRY, action, arg))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Your router ran the model's **real** choice safely &mdash; and would return `"unknown tool: ..."` if the model hallucinated a name.
- Splitting **deciding** (the model) from **executing** (the router) is exactly what makes agents debuggable.
- If the model picked `calculator` for a weather question, the router still runs cleanly &mdash; wrong tool, but no crash. You catch that by reading the trace.

## Your turn (open task &mdash; no grader)
Change `question` so the model should route to `calculator` or `lookup`, and re-run. Then force a failure:
call `route(REGISTRY, "delete_db", "x")` and `route(REGISTRY, "calculator", "hello")` directly. **What good
looks like:** every case returns a clean `{"ok": ..., "observation": ...}` dict &mdash; never a stack trace.

---
### Key takeaway
Routing turns a chosen action into a real observation -- safely. You fed a REAL model's choice through the dispatcher and it survived an unknown or failing tool.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>